In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.memory import ChatMessageHistory
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:

# RunnableWithMessageHistory는 대화를 직접 저장하지 않음, wrapper class
# BaseChatMessageHistory 인터페이스를 구현하는 클래스 객체에 대화 저장을 맡김

# BaseChatMessageHistory 인터페이스를 구현하는 클래스
 
# InMemoryChatMessageHistory: 애플리케이션의 메모리에 대화 기록을 저장
# FileChatMessageHistory: 로컬 파일에 대화 기록을 저장
# RedisChatMessageHistory: Redis 데이터베이스에 대화 기록을 저장
# SQLChatMessageHistory: MySQL, SQLite와 같은 관계형 SQL 데이터베이스에 대화 기록을 저장
# PostgresChatMessageHistory: PostgreSQL에 최적화된 대화 기록을 저장
# DynamoDBChatMessageHistory: AWS DynamoDB에 대화 기록을 저장

In [ ]:
# InMemoryChatMessageHistory == ChatMessageHistory (alias)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory() # []
    return store[session_id]

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 친절한 인공지능 비서야. 이전 대화 기록을 참고해서 답변해줘."),
        ("placeholder", "{history}"),
        ("human", "{input}"),
    ]
)

# RunnableWithMessageHistory 객체
chain = prompt | llm
with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
    # {'input': ■1, 'history': ■2}를 prompt에 전달
    # ■1는 "나의 이름은 철수야. 잘 부탁해"
    # ■2는 get_session_history(session_id) 리턴 값인 list of messages

    # HumanMessage로 변환된 ■1과 AIMessage로 변환된 llm의 응답을 InMemoryChatMessageHistory에 저장
    # RunnableWithMessageHistory는 어디에 저장할지의 정보를 get_session_history의 리턴으로 결정
)

session_id = "user_456"

# 첫 번째 대화
response1 = with_history.invoke(
    {"input": "나의 이름은 철수야. 잘 부탁해"},
    config={"configurable": {"session_id": session_id}}
)
print(f"첫 번째 응답: {response1.content}\n")

# 두 번째 대화 (이전 기록을 기억함)
response2 = with_history.invoke(
    {"input": "내가 방금 뭐라고 말했었지?"}, 
    config={"configurable": {"session_id": session_id}}
)
print(f"두 번째 응답: {response2.content}")

In [ ]:
# FileChatMessageHistory
# 대화 기록을 파일에 저장
# add_user_message()나 add_ai_message()를 호출하면 메모리에도 추가하고, 동시에 파일에도 저장
# 대화가 끝난 후 사용자의 입력과 AI의 응답을 FileChatMessageHistory 객체에 명시적으로 추가해야 함

# ChatMessageHistory는 메모리 기반으로서 프로그램이 종료되면 모든 기록이 사라짐

In [ ]:
# FileChatMessageHistory - 1

from langchain_core.messages import HumanMessage, AIMessage
from langchain.memory.chat_message_histories import FileChatMessageHistory
import os

file_path = "chat_history.json"

# 기존 파일이 있다면 삭제
if os.path.exists(file_path):
    os.remove(file_path)

# FileChatMessageHistory 객체 생성
chat_history = FileChatMessageHistory(file_path=file_path)

# 메시지 추가 > 파일에 저장, chat_history 내용도 업데이트
chat_history.add_user_message("안녕하세요! 제 이름은 밥입니다.")
chat_history.add_ai_message("안녕하세요, 밥님! 만나서 반갑습니다.")

for message in chat_history.messages:
    print(f"- {message.type.upper()}: {message.content}")

# 다음 셀에서 계속

In [ ]:

# 새로운 FileChatMessageHistory 객체를 생성(파일에서 기록 불러오기)
# > 애플리케이션을 재시작한 것과 같은 효과
reloaded_history = FileChatMessageHistory(file_path=file_path)

print("불러온 메시지:")
for message in reloaded_history.messages:
    print(f"- {message.type.upper()}: {message.content}")

# 다음 셀에서 계속

In [ ]:

# 대화 이어가기
print("\n두 번째 대화 시작...")
reloaded_history.add_user_message("제가 방금 뭐라고 말했죠?")
reloaded_history.add_ai_message("이름이 밥이라고 말씀하셨습니다.")

print("\n최신 파일에 저장된 메시지:")

reloaded_history_final = FileChatMessageHistory(file_path=file_path)
for message in reloaded_history_final.messages:
    print(f"- {message.type.upper()}: {message.content}")

In [ ]:
# FileChatMessageHistory - 2

import sys
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain.memory.chat_message_histories import FileChatMessageHistory
from langchain_core.messages import AIMessage

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
file_path = "chat_history_interactive.json"

chat_history = FileChatMessageHistory(file_path=file_path)

# 초기 메시지
if not chat_history.messages:
    chat_history.add_ai_message(AIMessage(content="안녕하세요! 무엇을 도와드릴까요?"))
    
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 이전 대화 기록을 참고하는 유용한 AI 비서입니다."),
    MessagesPlaceholder(variable_name="history"), # list of messages
    ("human", "{input}"),
])

# 체인 구성
# RunnablePassthrough.assign으로 'history' 변수를 명시적으로 추가
chain = (
    RunnablePassthrough.assign(
        # chat_history.messages에서 직접 메시지 리스트를 가져와 history 변수에 할당합니다.
        history=lambda x: chat_history.messages
    )
    # {"input": user_input, 'history': AIMessage(content="안녕하세요! 무엇을 도와드릴까요?")}
    | prompt
    | llm
)

# 6. 대화 루프 시작
print("챗봇과 대화하세요. (종료하려면 'exit'를 입력하세요)")
while True:
    try:
        user_input = input("User: ")
        if user_input.lower() == 'exit':
            print("대화를 종료합니다. 기록이 파일에 저장되었습니다.")
            break
        
        # 체인 호출
        response = chain.invoke({"input": user_input})
        print(f"AI: {response.content}")
        
        # 대화 기록을 저장
        # 체인 실행 후, 입력과 출력을 FileChatMessageHistory에 추가
        chat_history.add_user_message(user_input)
        chat_history.add_ai_message(response.content)

    except KeyboardInterrupt:
        print("\n대화를 강제로 종료합니다. 기록이 파일에 저장되었습니다.")
        sys.exit()

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain.memory.chat_message_histories import FileChatMessageHistory
from langchain_core.messages import AIMessage

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# 세션별로 다른 파일을 사용
def get_chat_history_by_session(session_id: str) -> FileChatMessageHistory:
    file_path = f"chat_history_{session_id}.json"
    history = FileChatMessageHistory(file_path=file_path)
    # 만약 파일이 없다면 새로운 파일을 만들고, history.messages는 빈 리스트
    if not history.messages:
        history.add_ai_message(AIMessage(content="안녕하세요! 무엇을 도와드릴까요?"))
        # prompt의 MessagesPlaceholder(variable_name="history")에 값 전달 가능
    return history

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 이전 대화 기록을 참고하는 유용한 AI 비서입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# 체인
# RunnablePassthrough.assign을 람다 함수를 사용하여 history를 가져옴
# {"input": user1_input, "session_id": user1_id}
chain = (
    # 입력은 {"input": "내 이름은 밥이야.", "session_id": user1_id}
    RunnablePassthrough.assign(
        history=lambda x: get_chat_history_by_session(x['session_id']).messages
    )
    # {"input": user1_input, "session_id": user1_id, 'history': []}
    | prompt
    | llm
)

# 대화 루프 시작
print("챗봇과 대화하세요. (종료하려면 'exit'를 입력하세요)")
print("---")

# 예시로 사용할 두 명의 사용자 세션 ID
user1_id = "user1"
user2_id = "user2"

# 첫 번째 사용자 대화
print(f"[{user1_id}] 님, 대화를 시작합니다.")
user1_history = get_chat_history_by_session(user1_id) # 해당 세션의 history 객체를 가져옴
user1_input = "내 이름은 밥이야."
response = chain.invoke({"input": user1_input, "session_id": user1_id})
print(f"AI: {response.content}")
# 대화 기록을 저장
user1_history.add_user_message(user1_input)
user1_history.add_ai_message(response.content)

# 두 번째 사용자 대화
print(f"\n[{user2_id}] 님, 대화를 시작합니다.")
user2_history = get_chat_history_by_session(user2_id)
user2_input = "내 이름은 앨리스야."
response = chain.invoke({"input": user2_input, "session_id": user2_id})
print(f"AI: {response.content}")
# 대화 기록을 저장
user2_history.add_user_message(user2_input)
user2_history.add_ai_message(response.content)

# 첫 번째 사용자로 돌아와 대화 이어가기
print(f"\n[{user1_id}] 님, 다시 돌아왔습니다.")
user1_input = "내가 좀 전에 뭐라고 말했지?"
response = chain.invoke({"input": user1_input, "session_id": user1_id})
print(f"AI: {response.content}")

# 두 번째 사용자로 돌아와 대화 이어가기
print(f"\n[{user2_id}] 님, 다시 돌아왔습니다.")
response = chain.invoke({"input": "내 이름이 뭐였지?", "session_id": user2_id})
print(f"AI: {response.content}")

In [ ]:
# batch

# asynchronous
# ainvoke
# abatch

In [ ]:
# batch
# 여러 입력(리스트 형태)에 대해 체인 또는 언어 모델을 동시에 실행
# chain.batch([{"product": "colorful socks"}, {"product": "eco-friendly water bottles"}])

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

prompt = ChatPromptTemplate.from_template("{product}을 만드는 회사의 좋은 이름은 무엇일까?")

# 체인
chain = prompt | llm

# 여러 개의 한글 입력 데이터 준비
products = [
    {"product": "친환경 물병"},
    {"product": "스마트 커피 머신"},
    {"product": "컬러풀한 양말"}
]

# batch 메서드 실행
# results는 각 입력에 대한 응답이 담긴 리스트
results = chain.batch(products)

# 5. 결과 출력
for i, result in enumerate(results):
    print('-'*100)
    print(i, result.content)